In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [2]:
train['TotalSpend'] = (
                        train['RoomService'].fillna(0) +
                        train['FoodCourt'].fillna(0) +
                        train['ShoppingMall'].fillna(0) +
                        train['Spa'].fillna(0) +
                        train['VRDeck'].fillna(0))

train[['Deck', 'Cabin_No','Side']] = train['Cabin'].str.split('/',expand=True)
train[['GroupID', 'MemberID']] = train['PassengerId'].str.split('_', expand=True)

group_counts = train['GroupID'].value_counts()
train["Group_Size"] = train["GroupID"].map(group_counts)

In [5]:
mask_spending = (train['CryoSleep'].isna()) & (train['TotalSpend'] > 0)
train.loc[mask_spending, 'CryoSleep'] = False

spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    train.loc[(train['CryoSleep'] == True) &(train[col].isna()), col] = 0

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
for col in cat_cols:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].mode()[0])

num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].median())

print("Remaining missing values:")
print(train.isna().sum().sum())

train.info()

Remaining missing values:
598
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8693 non-null   object 
 2   CryoSleep     8693 non-null   bool   
 3   Cabin         8494 non-null   object 
 4   Destination   8693 non-null   object 
 5   Age           8693 non-null   float64
 6   VIP           8693 non-null   bool   
 7   RoomService   8693 non-null   float64
 8   FoodCourt     8693 non-null   float64
 9   ShoppingMall  8693 non-null   float64
 10  Spa           8693 non-null   float64
 11  VRDeck        8693 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
 14  TotalSpend    8693 non-null   float64
 15  Deck          8693 non-null   object 
 16  Cabin_No      8494 non-null   object 
 17  Side          8693 non-null   object 
 18

In [6]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb

features_to_drop = ['Name', 'PassengerID', 'Cabin', 'GroupID', 'Member_ID']
model_df = train.drop(columns=[c for c in features_to_drop if c in train.columns], errors='ignore').copy()

model_df['Transported'] = model_df['Transported'].astype(int)

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

le = LabelEncoder()

for col in cat_cols:
    if col in model_df.columns:

        model_df[col] = model_df[col].astype(str)
        model_df[col] = le.fit_transform(model_df[col])

if 'Cabin_Num' in model_df.columns:
    model_df['Cabin_Num'] = pd.to_numeric(model_df['Cabin_Num'], errors='coerce').fillna(-1)

X = model_df.drop('Transported', axis=1)
y = model_df['Transported']

print("Features used: ", list(X.columns))

Features used:  ['PassengerId', 'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'Deck', 'Cabin_No', 'Side', 'MemberID', 'Group_Size']
